# Olist Brazilian E-Commerce — Exploratory Data Analysis

**Author:** [Lokanath Satapathy]
**Dataset:** [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle)

This notebook explores the Olist e-commerce dataset end-to-end — revenue trends, product performance, customer segmentation (RFM), retention (cohort analysis), and operations (delivery & payments). It is the Python/pandas companion to a parallel SQL + Power BI analysis of the same dataset, focused here on statistical exploration and visualization.

**Sections**
1. Setup & Data Loading
2. Data Overview & Quality Checks
3. Revenue & Growth Trends
4. Product Performance
5. Customer Segmentation (RFM)
6. Customer Retention (Cohort Analysis)
7. Operations — Delivery & Reviews
8. Operations — Geography
9. Operations — Payments
10. Summary of Findings


In [7]:
!pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)

DATA_DIR = "OneDrive/Documents/PW Skills/PW Skills/DataBase/projects/Brazilian E-Commerce Public Dataset by Olist/"  # <-- change this to your CSV folder path


In [13]:
orders = pd.read_csv(DATA_DIR + "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_DIR + "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_DIR + "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR + "olist_products_dataset.csv")
order_reviews = pd.read_csv(DATA_DIR + "olist_order_reviews_dataset.csv")
order_payments = pd.read_csv(DATA_DIR + "olist_order_payments_dataset.csv")
sellers = pd.read_csv(DATA_DIR + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(DATA_DIR + "product_category_name_translation.csv")

# Parse date columns
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Order items:", order_items.shape)
print("Products:", products.shape)
print("Reviews:", order_reviews.shape)
print("Payments:", order_payments.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'OneDrive/Documents/PW Skills/PW Skills/DataBase/projects/Brazilian E-Commerce Public Dataset by Olist/olist_orders_dataset.csv'

## 2. Data Overview & Quality Checks

In [ ]:
orders.head()

In [ ]:
orders["order_status"].value_counts()

In [ ]:
# Missing values check across key tables
for name, df in [("orders", orders), ("order_items", order_items),
                  ("products", products), ("order_reviews", order_reviews),
                  ("order_payments", order_payments)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"--- {name} ---")
    print(missing if len(missing) else "No missing values")
    print()


**Note:** `order_delivered_customer_date` will have missing values for orders that are cancelled/undelivered — these are excluded from delivery-time analysis later. We filter to `order_status == 'delivered'` for most revenue and delivery calculations to keep results business-meaningful.

In [ ]:
delivered = orders[orders["order_status"] == "delivered"].copy()
print(f"Delivered orders: {len(delivered):,} / {len(orders):,} total ({len(delivered)/len(orders)*100:.1f}%)")


## 3. Revenue & Growth Trends

### Insight — Platform Growth Trajectory
Monthly orders grew from a low base in early 2017 to a stable ~6,000–7,000/month range by 2018. Data before Jan 2017 reflects a pilot/launch phase (very low volume) and is treated as noise in growth-rate calculations below.

In [ ]:
order_rev = order_items.merge(delivered[["order_id", "order_purchase_timestamp"]], on="order_id")
order_rev["order_month"] = order_rev["order_purchase_timestamp"].dt.to_period("M").astype(str)

monthly = order_rev.groupby("order_month").agg(
    total_orders=("order_id", "nunique"),
    total_revenue=("price", lambda x: (x + order_rev.loc[x.index, "freight_value"]).sum())
).reset_index()

monthly.head()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["order_month"], monthly["total_revenue"], marker="o", color="#d62728")
ax.set_title("Monthly Revenue Trend (2016–2018)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (R$)")
ax.tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()


In [ ]:
# Month-over-month growth %, excluding the noisy 2016 pilot-phase months
monthly_clean = monthly[monthly["order_month"] >= "2017-02"].copy()
monthly_clean["mom_growth_pct"] = monthly_clean["total_revenue"].pct_change() * 100

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#2ca02c" if v >= 0 else "#d62728" for v in monthly_clean["mom_growth_pct"].fillna(0)]
ax.bar(monthly_clean["order_month"], monthly_clean["mom_growth_pct"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Month-over-Month Revenue Growth %", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Growth %")
ax.tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()


**Insight:** November 2017 shows the largest MoM spike, coinciding with Brazil's Black Friday — followed by a December cooldown (demand pull-forward). Growth flattens into a smaller oscillation band through 2018, suggesting market maturity (note: the Olist dataset is known to have incomplete records after Sep 2018).

## 4. Product Performance

In [ ]:
items_products = order_items.merge(products, on="product_id").merge(
    delivered[["order_id"]], on="order_id"
)

category_perf = items_products.groupby("product_category_name").agg(
    total_orders=("order_id", "nunique"),
    total_revenue=("price", "sum"),
    avg_price=("price", "mean")
).reset_index().sort_values("total_revenue", ascending=False).head(15)

category_perf


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=category_perf, y="product_category_name", x="total_revenue", palette="Reds_r", ax=ax)
ax.set_title("Top 15 Product Categories by Revenue", fontsize=14, fontweight="bold")
ax.set_xlabel("Total Revenue (R$)")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(
    category_perf["total_orders"], category_perf["avg_price"],
    s=category_perf["total_revenue"] / 3000, alpha=0.6, c=category_perf["total_revenue"],
    cmap="Reds"
)
for _, row in category_perf.iterrows():
    ax.annotate(row["product_category_name"], (row["total_orders"], row["avg_price"]), fontsize=8)
ax.set_title("Category Performance: Volume vs Avg Price (bubble = revenue)", fontsize=14, fontweight="bold")
ax.set_xlabel("Total Orders")
ax.set_ylabel("Average Price (R$)")
plt.tight_layout()
plt.show()


**Insight:** Health & Beauty and Watches/Gifts lead in revenue with relatively high average prices, while Bed/Bath/Table drives volume at lower price points — a classic volume-vs-value trade-off worth considering for bundling strategies.

## 5. Customer Segmentation (RFM Analysis)

Recency, Frequency, Monetary scoring — computed per `customer_unique_id` (a customer may have multiple `customer_id`s in Olist, one per order).

In [ ]:
cust_orders = delivered.merge(customers, on="customer_id")
order_value = order_items.groupby("order_id").apply(
    lambda x: (x["price"] + x["freight_value"]).sum()
).rename("order_value").reset_index()

cust_orders = cust_orders.merge(order_value, on="order_id")

snapshot_date = cust_orders["order_purchase_timestamp"].max()

rfm = cust_orders.groupby("customer_unique_id").agg(
    recency_days=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("order_value", "sum")
).reset_index()

rfm.head()


In [ ]:
rfm["r_score"] = pd.qcut(rfm["recency_days"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)

def segment(row):
    if row.r_score >= 3 and row.f_score >= 3 and row.m_score >= 3:
        return "Champions"
    elif row.r_score >= 3 and row.f_score >= 2:
        return "Loyal Customers"
    elif row.r_score >= 3 and row.f_score == 1:
        return "Recent Customers"
    elif row.r_score <= 2 and row.f_score >= 3:
        return "At Risk"
    elif row.r_score == 1 and row.f_score == 1 and row.m_score >= 3:
        return "Cant Lose Them"
    else:
        return "Others"

rfm["segment"] = rfm.apply(segment, axis=1)

segment_summary = rfm.groupby("segment").agg(
    num_customers=("customer_unique_id", "count"),
    avg_monetary=("monetary", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_recency=("recency_days", "mean")
).reset_index()
segment_summary["pct_of_total"] = (segment_summary["num_customers"] / segment_summary["num_customers"].sum() * 100).round(2)
segment_summary.sort_values("num_customers", ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
colors = sns.color_palette("Reds", len(segment_summary))
ax.pie(segment_summary["num_customers"], labels=segment_summary["segment"], autopct="%1.1f%%",
       colors=colors, startangle=90)
ax.set_title("Customer Segments (RFM)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


**Insight:** Champions are a small slice (~8%) of customers but the highest average spenders — a Pareto pattern. The "At Risk" segment is typically the largest group, representing the biggest win-back opportunity. Average frequency close to 1.0 across nearly every segment signals weak repeat-purchase behavior platform-wide — confirmed independently by the cohort analysis below.

## 6. Customer Retention (Cohort Analysis)

In [ ]:
cust_orders["order_month"] = cust_orders["order_purchase_timestamp"].dt.to_period("M")
cust_orders["cohort_month"] = cust_orders.groupby("customer_unique_id")["order_month"].transform("min")

def month_diff(row):
    return (row["order_month"].year - row["cohort_month"].year) * 12 + (row["order_month"].month - row["cohort_month"].month)

cust_orders["month_number"] = cust_orders.apply(month_diff, axis=1)

cohort_data = cust_orders.groupby(["cohort_month", "month_number"])["customer_unique_id"].nunique().reset_index()
cohort_sizes = cohort_data[cohort_data["month_number"] == 0][["cohort_month", "customer_unique_id"]].rename(
    columns={"customer_unique_id": "cohort_size"}
)

cohort_data = cohort_data.merge(cohort_sizes, on="cohort_month")
cohort_data["retention_pct"] = (cohort_data["customer_unique_id"] / cohort_data["cohort_size"] * 100).round(2)

# Keep only statistically meaningful cohorts (size >= 100) and exclude month 0 (always 100%, trivial)
cohort_pivot = cohort_data[cohort_data["cohort_size"] >= 100].pivot(
    index="cohort_month", columns="month_number", values="retention_pct"
)
cohort_pivot.index = cohort_pivot.index.astype(str)
cohort_pivot.head()


In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(cohort_pivot.drop(columns=[0]), annot=True, fmt=".1f", cmap="Reds", cbar_kws={"label": "Retention %"}, ax=ax)
ax.set_title("Monthly Cohort Retention Heatmap (Month 0 excluded — always 100%)", fontsize=14, fontweight="bold")
ax.set_xlabel("Months Since First Purchase")
ax.set_ylabel("Acquisition Cohort")
plt.tight_layout()
plt.show()


**Insight:** Every cohort collapses from 100% at Month 0 to under 1% retention by Month 1 — a cliff, not a gradual decline — and this pattern holds across every cohort regardless of acquisition size, pointing to a structural, platform-wide retention gap rather than a one-off or seasonal issue.

## 7. Operations — Delivery & Reviews

In [ ]:
delivery = delivered.dropna(subset=["order_delivered_customer_date"]).merge(
    order_reviews[["order_id", "review_score"]], on="order_id"
)
delivery["actual_delivery_days"] = (delivery["order_delivered_customer_date"] - delivery["order_purchase_timestamp"]).dt.days
delivery["days_early_or_late"] = (delivery["order_estimated_delivery_date"] - delivery["order_delivered_customer_date"]).dt.days

def bucket(days):
    if days < 0:
        return "Late Delivery"
    elif days <= 5:
        return "On-Time (0-5 days early)"
    else:
        return "Very Early (5+ days early)"

delivery["delivery_status"] = delivery["days_early_or_late"].apply(bucket)

delivery_summary = delivery.groupby("delivery_status").agg(
    total_orders=("order_id", "count"),
    avg_review_score=("review_score", "mean"),
    avg_delivery_days=("actual_delivery_days", "mean"),
    bad_review_pct=("review_score", lambda x: (x <= 2).mean() * 100)
).round(2).reindex(["Very Early (5+ days early)", "On-Time (0-5 days early)", "Late Delivery"])

delivery_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=delivery_summary.reset_index(), x="delivery_status", y="avg_review_score", palette="Reds_r", ax=axes[0])
axes[0].set_title("Avg Review Score by Delivery Timeliness")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=delivery_summary.reset_index(), x="delivery_status", y="bad_review_pct", palette="Reds", ax=axes[1])
axes[1].set_title("Negative Review Rate (%) by Delivery Timeliness")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


**Insight:** Late deliveries roughly halve the average review score and carry a negative-review rate several times higher than early/on-time orders — late orders also take substantially longer on average, suggesting a systemic logistics issue in a specific subset of orders rather than random variance.

## 8. Operations — Geography

In [ ]:
delivery_geo = delivery.merge(customers[["customer_id", "customer_state"]], on="customer_id")

state_summary = delivery_geo.groupby("customer_state").agg(
    total_orders=("order_id", "count"),
    avg_delivery_days=("actual_delivery_days", "mean"),
    late_pct=("days_early_or_late", lambda x: (x < 0).mean() * 100)
).round(2)

state_summary = state_summary[state_summary["total_orders"] >= 100].sort_values("avg_delivery_days", ascending=False).head(15)
state_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=state_summary.reset_index(), y="customer_state", x="avg_delivery_days", palette="Reds_r", ax=ax)
ax.set_title("Slowest Delivery States (avg days, min 100 orders)", fontsize=14, fontweight="bold")
ax.set_xlabel("Avg Delivery Days")
ax.set_ylabel("State")
plt.tight_layout()
plt.show()


**Insight:** The slowest-delivery states are concentrated in North/Northeast Brazil (e.g., Amazonas, Alagoas, Pará), 2–3x slower than the national benchmark — reflecting Brazil's well-documented Southeast/North-Northeast infrastructure gap.

## 9. Operations — Payments

In [ ]:
payments = order_payments.merge(delivered[["order_id"]], on="order_id").merge(
    order_reviews[["order_id", "review_score"]], on="order_id"
)

payment_summary = payments.groupby("payment_type").agg(
    total_orders=("order_id", "count"),
    avg_order_value=("payment_value", "mean"),
    avg_installments=("payment_installments", "mean"),
    avg_review_score=("review_score", "mean")
).round(2).sort_values("total_orders", ascending=False)

payment_summary


In [ ]:
cc_installments = order_payments[order_payments["payment_type"] == "credit_card"].groupby(
    "payment_installments"
)["payment_value"].mean().reset_index().rename(columns={"payment_value": "avg_order_value"})

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cc_installments["payment_installments"], cc_installments["avg_order_value"], marker="o", color="#d62728")
ax.set_title("Avg Order Value vs Installment Count (Credit Card)", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Installments")
ax.set_ylabel("Avg Order Value (R$)")
plt.tight_layout()
plt.show()


**Insight:** Credit card dominates payment share, and average order value rises consistently with installment count — installments function as an affordability/conversion lever for higher-ticket purchases in the Brazilian market.

## 10. Summary of Findings

1. **Growth:** Orders scaled from ~750/month (Jan 2017) to a stable ~6-7K/month (2018); clear Black Friday seasonal spike in Nov 2017.
2. **Products:** Health & Beauty and Watches/Gifts are high-margin categories; Bed/Bath/Table is volume-driven but lower-margin.
3. **RFM:** Champions (~8% of customers) drive disproportionate revenue; "At Risk" is typically the largest segment — a major win-back opportunity.
4. **Retention:** Month-1 retention is below 1% across every cohort — Olist behaves as a largely one-time-purchase marketplace.
5. **Delivery:** Late deliveries roughly halve review scores and carry a much higher negative-review rate.
6. **Geography:** North/Northeast Brazil faces significantly longer delivery times, reflecting infrastructure gaps.
7. **Payments:** Credit card dominates; installments enable higher-value purchases.

**Recommendations:** Prioritize retention campaigns for the "At Risk" segment, audit logistics for late-delivery outliers (especially North/Northeast routes), and promote installment options for high-ticket categories.
